In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter

# ------------------------------------------------------------
# 1. Load and inspect the source data
# ------------------------------------------------------------
INPUT_FILE = "ProteinX_Sales_Marketing.csv"
df = pd.read_csv(INPUT_FILE)

print("Original shape:", df.shape)
print("\nMissing values:")
print(df.isna().sum())
print("\nOriginal data:")
print(df.to_string(index=False))

# ------------------------------------------------------------
# 2. Clean the data
# ------------------------------------------------------------
# Treat the exact repeated LinkedIn row as an accidental duplicate.
# Confirm this assumption against transaction IDs if available.
duplicate_rows = df.loc[df.duplicated(keep="first")].copy()
clean = df.drop_duplicates().copy()

# Dates are day-month-year.
clean["Date"] = pd.to_datetime(clean["Date"], format="%d-%m-%Y")

# Consolidate channel spelling variants.
clean["Marketing Channel"] = (
    clean["Marketing Channel"]
    .str.strip()
    .replace({
        "Instgram": "Instagram",
        "instagram": "Instagram",
        "Face book": "Facebook",
        "FB": "Facebook",
    })
)

# Parse the single "$60" entry as 60.
# Assume all monetary entries share a denomination; currency is unspecified.
# Missing spend remains missing, not zero.
clean["Ad Spend"] = pd.to_numeric(
    clean["Ad Spend"].str.replace("$", "", regex=False),
    errors="coerce",
)

clean["Product Sold"] = (
    clean["Product Sold"]
    .replace({"Proteinx Lite": "ProteinX Lite"})
    .fillna("Unspecified (failed)")
)

clean["Month"] = clean["Date"].dt.to_period("M").astype(str)

# This is not profit: product costs and other expenses are unavailable.
clean["Revenue less known ad spend"] = (
    clean["Revenue"] - clean["Ad Spend"]
)

# Keep failed activity in spend totals.
# Use matched records for ROAS so missing spend is not implicitly zero.
known = clean.loc[clean["Ad Spend"].notna()].copy()
paid = known.loc[known["Ad Spend"] > 0].copy()

# ------------------------------------------------------------
# 3. Channel analysis
# ------------------------------------------------------------
channel = clean.groupby("Marketing Channel").agg(
    Records=("Revenue", "size"),
    Units=("Units", "sum"),
    Revenue=("Revenue", "sum"),
    Known_ad_spend=("Ad Spend", lambda x: x.sum(min_count=1)),
    Missing_spend=("Ad Spend", lambda x: x.isna().sum()),
)

channel["Revenue_with_known_spend"] = (
    known.groupby("Marketing Channel")["Revenue"].sum()
)

# Direct has zero recorded spend, so its ROAS is undefined.
channel["Matched_ROAS"] = (
    channel["Revenue_with_known_spend"]
    / channel["Known_ad_spend"].replace(0, np.)
)

# This difference is not profit and is overstated where spend is missing.
channel["Revenue_minus_known_spend"] = (
    channel["Revenue"] - channel["Known_ad_spend"]
)

# ------------------------------------------------------------
# 4. Monthly analysis
# ------------------------------------------------------------
monthly = clean.groupby("Month").agg(
    Records=("Revenue", "size"),
    Units=("Units", "sum"),
    Revenue=("Revenue", "sum"),
    Known_ad_spend=("Ad Spend", "sum"),
    Missing_spend=("Ad Spend", lambda x: x.isna().sum()),
)

# Includes Direct revenue, so this is not paid-channel ROAS.
monthly["Matched_ROAS"] = (
    known.groupby("Month")["Revenue"].sum()
    / monthly["Known_ad_spend"].replace(0, np.)
)

# Paid-only monthly ROAS excludes Direct and missing-spend records.
paid_month = paid.groupby("Month").agg(
    Revenue=("Revenue", "sum"),
    Ad_spend=("Ad Spend", "sum"),
)

paid_month["ROAS"] = (
    paid_month["Revenue"]
    / paid_month["Ad_spend"].replace(0, np.)
)

# ------------------------------------------------------------
# 5. Product analysis
# ------------------------------------------------------------
product = clean.groupby("Product Sold").agg(
    Records=("Revenue", "size"),
    Units=("Units", "sum"),
    Revenue=("Revenue", "sum"),
)

product["Revenue_per_unit"] = (
    product["Revenue"] / product["Units"].replace(0, np.)
)

product_channel = clean.pivot_table(
    index="Marketing Channel",
    columns="Product Sold",
    values="Revenue",
    aggfunc="sum",
    fill_value=0,
)

# ------------------------------------------------------------
# 6. Print results and verification checks
# ------------------------------------------------------------
print("\nRemoved assumed duplicate:")
print(duplicate_rows.to_string(index=False))

print("\nCleaned totals:")
print(clean[["Units", "Revenue", "Ad Spend"]].sum())

print("\nChannel analysis:")
print(channel.to_string())

print("\nMonthly analysis (matched ROAS includes Direct):")
print(monthly.to_string())

print("\nPaid-only monthly analysis:")
print(paid_month.to_string())

print("\nProduct analysis:")
print(product.to_string())

print("\nChannel/product revenue:")
print(product_channel.to_string())

total_known_spend = known["Ad Spend"].sum()
paid_roas = paid["Revenue"].sum() / paid["Ad Spend"].sum()
matched_all_channel_ratio = known["Revenue"].sum() / total_known_spend

print(f"\nRecorded paid-channel ROAS: {paid_roas:.4f}x")
print(
    "Matched all-channel revenue / known spend "
    f"(includes Direct): {matched_all_channel_ratio:.4f}x"
)

# ------------------------------------------------------------
# 7. Create the four-panel Matplotlib dashboard
# ------------------------------------------------------------
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "axes.titlepad": 16,
})

navy = "#17324D"
teal = "#009C95"
red = "#D75D58"
background = "#F7F9FC"

fig, axs = plt.subplots(2, 2, figsize=(16, 11.5))
fig.patch.set_facecolor(background)

fig.suptitle(
    "ProteinX | Sales & Marketing Performance",
    fontsize=23,
    fontweight="bold",
    color=navy,
    y=0.985,
)

fig.text(
    0.5,
    0.945,
    "1 Aug–5 Nov 2026 • 16 records after removing one assumed duplicate "
    "• monetary units as provided",
    ha="center",
    color="#526171",
    fontsize=11,
)

# Panel 1: Channel revenue versus recorded ad spend
ax = axs[0, 0]
c = channel.sort_values("Revenue", ascending=True)
y = np.arange(len(c))

ax.barh(
    y + 0.18,
    c["Revenue"],
    height=0.34,
    color=navy,
    label="Revenue",
)
ax.barh(
    y - 0.18,
    c["Known_ad_spend"],
    height=0.34,
    color=teal,
    label="Known ad spend",
)

ax.set_yticks(y, c.index)
ax.set_title("Channel revenue vs. recorded ad spend", loc="left")
ax.legend(frameon=False, loc="lower right")
ax.set_xlabel("Monetary units")
ax.xaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))

# Panel 2: Paid-channel ROAS
ax = axs[0, 1]
c = channel.loc[
    ["Facebook", "Instagram", "Website", "LinkedIn"]
].sort_values("Matched_ROAS")

bars = ax.barh(
    c.index,
    c["Matched_ROAS"],
    color=[red if value < 1 else teal for value in c["Matched_ROAS"]],
)

ax.axvline(1, color="#677383", linestyle="--", linewidth=1)
ax.set_xlim(0, 2.2)

for bar, value in zip(bars, c["Matched_ROAS"]):
    ax.text(
        value + 0.04,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.2f}×",
        va="center",
        fontweight="bold",
    )

ax.set_title("Recorded paid-channel ROAS", loc="left")
ax.set_xlabel("Revenue / ad spend • 1× covers ads only")
ax.text(
    0,
    -0.24,
    "Instagram excludes the record with unknown spend. "
    "Direct has zero recorded spend.",
    transform=ax.transAxes,
    fontsize=9,
    color="#526171",
    wrap=True,
)

# Panel 3: Revenue by product
ax = axs[1, 0]
p = product.loc[product["Revenue"] > 0].sort_values("Revenue")

bars = ax.barh(
    p.index.str.replace("ProteinX ", "", regex=False),
    p["Revenue"],
    color=navy,
)

for bar, value in zip(bars, p["Revenue"]):
    ax.text(
        value + 25,
        bar.get_y() + bar.get_height() / 2,
        f"{value:,.0f}  ({value / clean['Revenue'].sum():.1%})",
        va="center",
        fontsize=10,
    )

ax.set_xlim(0, 2500)
ax.set_title(
    "Loaded and Bulk Powder deliver 85% of revenue",
    loc="left",
)
ax.set_xlabel("Revenue • shares of total revenue")

# Panel 4: Monthly revenue and recorded spend
ax = axs[1, 1]
x = np.arange(len(monthly))

revenue_bars = ax.bar(
    x - 0.18,
    monthly["Revenue"],
    0.34,
    color=navy,
    label="All revenue",
)
spend_bars = ax.bar(
    x + 0.18,
    monthly["Known_ad_spend"],
    0.34,
    color=teal,
    label="Known ad spend",
)

for bars in (revenue_bars, spend_bars):
    for bar in bars:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 20,
            f"{bar.get_height():,.0f}",
            ha="center",
            fontsize=9,
        )

# November only contains November 1–5.
for bar in (revenue_bars[-1], spend_bars[-1]):
    bar.set_alpha(0.45)
    bar.set_hatch("//")

ax.set_xticks(x, ["Aug", "Sep", "Oct", "Nov 1–5*"])
ax.set_ylim(0, 1550)
ax.set_title("Monthly revenue and spend", loc="left")
ax.legend(frameon=False, fontsize=9)
ax.set_ylabel("Monetary units")

fig.text(
    0.04,
    0.025,
    "Notes: Missing spend is not imputed. Failed activity is retained. "
    "November is partial; earlier months reflect supplied records, "
    "not verified completeness.\n"
    "ROAS is attribution-based, not causal return or profit. "
    'Currency is unspecified; the single "$60" entry was parsed as 60.',
    fontsize=10,
    color="#526171",
)

for i, ax in enumerate(axs.flat):
    ax.set_facecolor(background)
    ax.grid(axis="y" if i == 3 else "x", alpha=0.12)
    ax.set_axisbelow(True)

fig.subplots_adjust(
    left=0.14,
    right=0.96,
    top=0.885,
    bottom=0.13,
    hspace=0.58,
    wspace=0.38,
)

fig.savefig(
    "ProteinX_Performance_Dashboard.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()

# ------------------------------------------------------------
# 8. Export the analysis workbook
# Requires openpyxl: pip install openpyxl
# ------------------------------------------------------------
notes = pd.DataFrame({
    "Analysis notes": [
        "Dates interpreted as DD-MM-YYYY; coverage 2026-08-01 "
        "through 2026-11-05.",
        "One exact repeated LinkedIn row dated 2026-08-15 removed "
        "as an assumed duplicate; confirmation requires transaction IDs.",
        "Removed duplicate: 5 units, 400 revenue, 300 spend. "
        "Raw totals: 485 units, 4060 revenue, 2460 known spend.",
        "Channel spelling variants consolidated; Proteinx Lite "
        "standardized to ProteinX Lite.",
        "Single $60 spend entry parsed as 60; assume common monetary "
        "denomination; currency not established.",
        "Instagram 2026-08-02: spend missing, 10 units and 60 revenue. "
        "Spend not imputed.",
        "Failed Facebook record kept: 200 spend, zero revenue, "
        "product unspecified.",
        "Known positive-spend ROAS = 2710 / 2160 = 1.2546; "
        "excludes Direct and unknown-spend record.",
        "Matched all-channel revenue/spend = 3600 / 2160 = 1.6667; "
        "includes Direct revenue; not paid ROAS.",
        "Total revenue minus known ad spend = 1500, before missing "
        "spend, COGS, fulfilment and other costs; not profit.",
        "ROAS uses revenue only from records with known spend; "
        "zero-spend Direct ROAS undefined.",
        "No customer IDs, transaction IDs, conversions, impressions, "
        "clicks, costs or attribution windows provided.",
        "November partial. No completeness verification for other "
        "months. Small sample prevents causal or statistically "
        "robust conclusions.",
    ]
})

with pd.ExcelWriter(
    "ProteinX_Analysis.xlsx",
    engine="openpyxl",
) as writer:
    df.to_excel(writer, sheet_name="Original data", index=False)
    clean.to_excel(writer, sheet_name="Cleaned data", index=False)
    channel.to_excel(writer, sheet_name="Channel analysis")
    product.to_excel(writer, sheet_name="Product analysis")
    monthly.to_excel(writer, sheet_name="Monthly analysis")
    paid_month.to_excel(writer, sheet_name="Paid monthly ROAS")
    notes.to_excel(
        writer,
        sheet_name="Assumptions and quality",
        index=False,
    )

    for worksheet in writer.book.worksheets:
        worksheet.freeze_panes = "B2"
        worksheet.auto_filter.ref = worksheet.dimensions

        for column in worksheet.columns:
            letter = column[0].column_letter
            max_length = max(
                len(str(cell.value)) if cell.value is not None else 0
                for cell in column
            )
            worksheet.column_dimensions[letter].width = min(
                90, max(15, max_length + 2)
            )

print("\nDashboard and analysis workbook saved successfully.")